In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# 1. パラメータ設定 (対称な設定)
params = {
    'r': [1.0, 1.0, 1.0, 1.0],      # [rx1, ry1, rx2, ry2]
    'lambda_diag': [0.5, 0.5],      # [l11, l22]
    'c': [0.8, 0.8],                # [c1, c2]
    'd': [0.8, 0.8]                 # [d1, d2]
}

rx1, ry1, rx2, ry2 = params['r']
l11, l22 = params['lambda_diag']
c1, c2 = params['c']
d1, d2 = params['d']

# 2. メッシュグリッド作成 (lambda_12 vs lambda_21)
res = 400
l12_vals = np.linspace(0, 1.0, res)
l21_vals = np.linspace(0, 1.0, res)
L12, L21 = np.meshgrid(l12_vals, l21_vals)

# 3. 平衡点の計算
# 行列式 Delta_x, Delta_y
Det_x = l11 * l22 - L12 * L21
Det_y = c1 * d2 * l11 * l22 - c2 * d1 * L12 * L21

# 分子 (Numerators) の計算
# x1分子: l22*ry1 - l12*ry2
Num_x1 = l22 * ry1 - L12 * ry2
# x2分子: l11*ry2 - l21*ry1
Num_x2 = l11 * ry2 - L21 * ry1
# y1分子: d2*l22*rx1 - d1*l21*rx2
Num_y1 = d2 * l22 * rx1 - d1 * L21 * rx2
# y2分子: c1*l11*rx2 - c2*l12*rx1
Num_y2 = c1 * l11 * rx2 - c2 * L12 * rx1

# 平衡点 (分母で割る)
# ゼロ除算回避のためごく小さな値を加えるか、マスク処理するが、今回はそのまま計算してinfは無視
X1 = Num_x1 / Det_x
X2 = Num_x2 / Det_x
Y1 = Num_y1 / Det_y
Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け (整数IDを振る)
# ID 0: その他 (複数負など複雑な領域)
# ID 1: Coexistence (All > 0)
# ID 2: x1 Extinct (x1 < 0, others > 0)
# ID 3: x2 Extinct (x2 < 0, others > 0)
# ID 4: Prey Collapse (y1 < 0 or y2 < 0)

Category = np.zeros_like(X1, dtype=int) # デフォルト0

# マスク作成
pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

# 条件判定
# 1. 全て正 (共存)
mask_coexist = pos_x1 & pos_x2 & pos_y1 & pos_y2
Category[mask_coexist] = 1

# 2. x1のみ負 (x1絶滅, x2は生存)
mask_x1_die = (~pos_x1) & pos_x2 & pos_y1 & pos_y2
Category[mask_x1_die] = 2

# 3. x2のみ負 (x2絶滅, x1は生存)
mask_x2_die = pos_x1 & (~pos_x2) & pos_y1 & pos_y2
Category[mask_x2_die] = 3

# 4. 被食者が負 (y1 < 0 or y2 < 0) -> 系として成立しない/崩壊
# ※モデル上ではyが負になるのはxによる捕食圧またはd項の影響が強すぎる場合
mask_y_die = (~pos_y1) | (~pos_y2)
Category[mask_y_die] = 4

# 5. プロット
fig, ax = plt.subplots(figsize=(10, 8))

# カラーマップ設定
# 0:Gray, 1:Yellow, 2:Blue, 3:Red, 4:Black
colors = ['lightgray', 'gold', 'skyblue', 'salmon', '#444444']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

im = ax.imshow(Category, extent=[0, 1.0, 0, 1.0], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# 境界線の描画 (理論的な境界)
# x1分子=0の線: l12 = l22 * ry1 / ry2 = 0.5 * 1 / 1 = 0.5
ax.axvline(x=0.5, color='blue', linestyle='--', linewidth=1.5, label='$x_1=0$ Boundary')

# x2分子=0の線: l21 = l11 * ry2 / ry1 = 0.5 * 1 / 1 = 0.5
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='$x_2=0$ Boundary')

# y分子=0の線 (今回のパラメータではx分子と同じ0.5になるが、一般化のため記述)
# y1=0: l21 = (d2*l22*rx1)/(d1*rx2) = (0.8*0.5*1)/(0.8*1) = 0.5
# y2=0: l12 = 0.5
# 今回はxの境界と重なるため図示省略

# 凡例 (Custom Legend)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence (All > 0)'),
    Patch(facecolor='skyblue', edgecolor='k', label='$x_1 < 0$ (Species 1 Predator Extinct)'),
    Patch(facecolor='salmon', edgecolor='k', label='$x_2 < 0$ (Species 2 Predator Extinct)'),
    Patch(facecolor='#444444', edgecolor='k', label='$y < 0$ (Prey Collapse/Unstable)'),
    Patch(facecolor='lightgray', edgecolor='k', label='Mixed Negatives / Invalid')
]

ax.legend(handles=legend_elements, loc='upper right')
ax.set_xlabel('Cross-Interaction $\lambda_{12}$ ($x_2$ eats $y_1$)')
ax.set_ylabel('Cross-Interaction $\lambda_{21}$ ($x_1$ eats $y_2$)')
ax.set_title('Stability Regions by Equilibrium Signs\n(Parameter Space: $\lambda_{12}$ vs $\lambda_{21}$)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定
params = {
    'r': [1.5, 1.0, 1.0, 1.0],      # [rx1, ry1, rx2, ry2]
    'lambda_diag': [0.5, 0.5],      # [lambda11, lambda22]
    'c': [0.8, 0.8],                # [c1, c2]
    'd': [0.8, 0.8]                 # [d1, d2]
}

rx1, ry1, rx2, ry2 = params['r']
l11, l22 = params['lambda_diag']
c1, c2 = params['c']
d1, d2 = params['d']

# 2. メッシュグリッド
res = 400
l12_vals = np.linspace(0, 1.0, res)
l21_vals = np.linspace(0, 1.0, res)
L12, L21 = np.meshgrid(l12_vals, l21_vals)

# 3. 平衡点の計算 (x=Prey, y=Predator)
# --- X (Prey: 被食者) ---
Det_x = c1 * d2 * l11 * l22 - c2 * d1 * L12 * L21
Num_x1 = d2 * l22 * rx1 - d1 * L21 * rx2
Num_x2 = c1 * l11 * rx2 - c2 * L12 * rx1

# --- Y (Predator: 捕食者) ---
Det_y = l11 * l22 - L12 * L21
Num_y1 = l22 * ry1 - L12 * ry2
Num_y2 = l11 * ry2 - L21 * ry1

with np.errstate(divide='ignore', invalid='ignore'):
    X1 = Num_x1 / Det_x
    X2 = Num_x2 / Det_x
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け
Category = np.zeros_like(X1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

# Base Layer (単独死滅)
Category[~pos_y1] = 2
Category[~pos_y2] = 3
Category[~pos_x1] = 4
Category[~pos_x2] = 5

# Middle Layer (両方死滅)
mask_both_y = (~pos_y1) & (~pos_y2)
Category[mask_both_y] = 10
mask_both_x = (~pos_x1) & (~pos_x2)
Category[mask_both_x] = 11

# Top Layer (複合条件)
mask_green = (~pos_y2) & (~pos_x2)
Category[mask_green] = 6
mask_gray = (~pos_y1) & (~pos_x1)
Category[mask_gray] = 7
mask_red = (~pos_y2) & (~pos_x1)
Category[mask_red] = 8
mask_blue = (~pos_y1) & (~pos_x2)
Category[mask_blue] = 9

# Coexistence
mask_coexist = pos_x1 & pos_x2 & pos_y1 & pos_y2
Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(12, 10))

colors = [
    'white', 'gold', 
    'skyblue', 'salmon', 'SlateBlue', 'IndianRed', 
    'MediumSeaGreen', 'lightgray', 'red', 'blue', 
    'purple', '#444444'
]
cmap = ListedColormap(colors)
norm = BoundaryNorm(np.arange(-0.5, 12.5, 1), cmap.N)

im = ax.imshow(Category, extent=[0, 1.0, 0, 1.0], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# --- 曲線 (境界線) 描画 ---
x_line = np.linspace(0.001, 1.0, 500)

# 1. Singularity Det_y = 0 (白色の実線)
k_y = l11 * l22
y_dety = k_y / x_line
ax.plot(x_line, y_dety, color='white', linestyle='-', linewidth=2.5, label=r'Singularity ($Det_y=0$)')

# 2. Singularity Det_x = 0 (黒色の破線)
k_x = (c1 * d2 * l11 * l22) / (c2 * d1)
y_detx = k_x / x_line
ax.plot(x_line, y_detx, color='black', linestyle='--', linewidth=2.5, label=r'Singularity ($Det_x=0$)')

# 3. 分子ゼロ線
ax.axvline(x=0.5, color='skyblue', linestyle=':', linewidth=1.5, label=r'$y_1=0$ Boundary')
ax.axhline(y=0.5, color='salmon', linestyle=':', linewidth=1.5, label=r'$y_2=0$ Boundary')

# --- パラメータ表示 (ここを追加) ---
textstr = '\n'.join((
    r'Fixed Parameters:',
    r'$r = [%.1f, %.1f, %.1f, %.1f]$' % tuple(params['r']),
    r'$\lambda_{11}, \lambda_{22} = %.1f, %.1f$' % tuple(params['lambda_diag']),
    r'$c = [%.1f, %.1f]$' % tuple(params['c']),
    r'$d = [%.1f, %.1f]$' % tuple(params['d'])
))

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

# 凡例
legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence'),
    
    # 複合条件
    Patch(facecolor='MediumSeaGreen', edgecolor='k', label='$y_2, x_2 < 0$ (Green)'),
    Patch(facecolor='lightgray', edgecolor='k', label='$y_1, x_1 < 0$ (Gray)'),
    Patch(facecolor='red', edgecolor='k', label='$y_2, x_1 < 0$ (Red)'),
    Patch(facecolor='blue', edgecolor='k', label='$y_1, x_2 < 0$ (Blue)'),

    # 単独・その他
    Patch(facecolor='skyblue', edgecolor='k', label='Only $y_1 < 0$'),
    Patch(facecolor='salmon', edgecolor='k', label='Only $y_2 < 0$'),
    Patch(facecolor='purple', edgecolor='k', label='Both $y < 0$'),
    
    # 線
    Line2D([0], [0], color='white', lw=2.5, linestyle='-', label=r'Singularity ($Det_y$)'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='--', label=r'Singularity ($Det_x$)')
]

ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
ax.set_xlabel('Cross-Interaction $\lambda_{12}$')
ax.set_ylabel('Cross-Interaction $\lambda_{21}$')
ax.set_title(r'Stability Regions ')
ax.grid(True, linestyle=':', alpha=0.5)
ax.set_xlim(0, 1.0)
ax.set_ylim(0, 1.0)


plt.savefig("a.png",dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定
params = {
    'r': [1.0, 1.0, 1.0, 1.0],      # [rx1, ry1, rx2, ry2]
    'lambda_offdiag': [0.3, 0.3],   # [lambda12, lambda21] (定数)
    'c': [0.8, 0.6],                # [c1, c2]
    'd': [0.6, 0.8]                 # [d1, d2]
}

rx1, ry1, rx2, ry2 = params['r']
l12, l21 = params['lambda_offdiag']
c1, c2 = params['c']
d1, d2 = params['d']

# 2. メッシュグリッド
res = 400
vals = np.linspace(0.001, 1.0, res)
L11, L22 = np.meshgrid(vals, vals)

# 3. 計算
# L11, L22を変数、l12, l21を定数として計算
Det_x = L11 * L22 - l12 * l21
Det_y = c1 * d2 * L11 * L22 - c2 * d1 * l12 * l21

Num_x1 = L22 * ry1 - l12 * ry2
Num_x2 = L11 * ry2 - l21 * ry1
Num_y1 = d2 * L22 * rx1 - d1 * l21 * rx2
Num_y2 = c1 * L11 * rx2 - c2 * l12 * rx1

# 平衡点の値を計算 (ゼロ除算は無視)
with np.errstate(divide='ignore', invalid='ignore'):
    X1 = Num_x1 / Det_x
    X2 = Num_x2 / Det_x
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け
Category = np.zeros_like(X1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

mask_y_die = (~pos_y1) | (~pos_y2)
Category[mask_y_die] = 4
mask_x1_die = (~pos_x1) & pos_x2
Category[mask_x1_die] = 2
mask_x2_die = pos_x1 & (~pos_x2)
Category[mask_x2_die] = 3
mask_coexist = pos_x1 & pos_x2 & pos_y1 & pos_y2
Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['white', 'gold', 'skyblue', 'salmon', '#444444']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

im = ax.imshow(Category, extent=[0, 1.0, 0, 1.0], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# --- 曲線描画 ---
x_line = np.linspace(0.001, 1.0, 500)
y_line1 = (l12 * l21) / x_line

numerator_curve = c2 * d1 * l12 * l21
denominator_curve = c1 * d2
y_line2 = numerator_curve / (denominator_curve * x_line)

ax.plot(x_line, y_line1, color='white', linestyle='-', linewidth=2.5, label=r'$\lambda_{11}\lambda_{22} = \lambda_{12}\lambda_{21}$')
ax.plot(x_line, y_line2, color='black', linestyle='--', linewidth=2.5, label=r'$C_1 d_2 \lambda_{11}\lambda_{22} = C_2 d_1 \lambda_{12}\lambda_{21}$')

# --- パラメータ表示 (修正箇所) ---
# ここで tuple() を使わず、単純な (c1, c2) のタプル形式を使用します
textstr = '\n'.join((
    r'Fixed Parameters:',
    r'$r = [%.1f, %.1f, %.1f, %.1f]$' % tuple(params['r']),
    r'$\lambda_{12}, \lambda_{21} = %.1f, %.1f$' % (l12, l21),
    r'$c = [%.1f, %.1f]$' % (c1, c2),
    r'$d = [%.1f, %.1f]$' % (d1, d2)
))

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

# 軸ラベルとタイトル
ax.set_ylim(0, 1.0)
ax.set_xlabel(r'Intraspecific Competition $\lambda_{11}$')
ax.set_ylabel(r'Intraspecific Competition $\lambda_{22}$')
ax.set_title('Phase Diagram: Stability & Coexistence')
ax.grid(True, linestyle=':', alpha=0.5)

legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence'),
    Patch(facecolor='skyblue', edgecolor='k', label='$x_1 < 0$ (x1 Extinct)'),
    Patch(facecolor='salmon', edgecolor='k', label='$x_2 < 0$ (x2 Extinct)'),
    Patch(facecolor='#444444', edgecolor='k', label='$y < 0$ (Prey Collapse)'),
    Line2D([0], [0], color='white', lw=2.5, label=r'Singularity (Det X)'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='--', label=r'Singularity (Det Y)')
]

ax.legend(handles=legend_elements, loc='upper right')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# 1. パラメータ設定 (lambdaは安定領域 0.2 に固定)
params = {
    'r': [1.0, 1.0, 1.0, 1.0],
    'lambda_matrix': [[0.5, 0.2],   # l12, l21 = 0.2 (安全圏)
                      [0.2, 0.5]],
    'c2': 0.8,                      # c2 固定
    'd2': 0.8                       # d2 固定
}

rx1, ry1, rx2, ry2 = params['r']
l11, l12 = params['lambda_matrix'][0]
l21, l22 = params['lambda_matrix'][1]
c2 = params['c2']
d2 = params['d2']

# 2. メッシュグリッド (c1 vs d1 を動かす)
res = 400
c1_vals = np.linspace(0, 2.0, res) # x1がy1を食べる効率
d1_vals = np.linspace(0, 2.0, res) # x1がy2を食べる効率
C1, D1 = np.meshgrid(c1_vals, d1_vals)

# 3. 平衡点の計算
# x* は c, d に依存しないため定数 (常に正)
Det_x = l11 * l22 - l12 * l21
x1_star = (l22 * ry1 - l12 * ry2) / Det_x # > 0
x2_star = (l11 * ry2 - l21 * ry1) / Det_x # > 0

# y* は c1, d1 に依存 (メッシュ計算)
Det_y = C1 * d2 * l11 * l22 - c2 * D1 * l12 * l21
Num_y1 = rx1 * (d2 * l22) - rx2 * (D1 * l21)
Num_y2 = rx2 * (C1 * l11) - rx1 * (c2 * l12)

# ゼロ除算無視
with np.errstate(divide='ignore', invalid='ignore'):
    # xは定数だが配列形状を合わせるためにbroadcast
    X1 = np.full_like(C1, x1_star)
    X2 = np.full_like(C1, x2_star)
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け (構造はそのまま、対象をyに変更)
Category = np.zeros_like(Y1, dtype=int)

# 判定条件
pos_y1 = (Y1 > 0) & np.isfinite(Y1)
pos_y2 = (Y2 > 0) & np.isfinite(Y2)

# 優先順位 1 (最下層): 片方が絶滅 (Blue: y1絶滅)
mask_y1_die = (~pos_y1)
Category[mask_y1_die] = 2 # Blue

# 優先順位 2 (上書き): もう片方が絶滅 (Red: y2絶滅)
mask_y2_die = (~pos_y2)
Category[mask_y2_die] = 3 # Red/Pink

# 優先順位 3 (上書き): 両方絶滅 (Gray)
# 重なっている部分をグレーにする
mask_both_die = (~pos_y1) & (~pos_y2)
Category[mask_both_die] = 4 # Gray

# 優先順位 4 (最上位): 共存 (Yellow)
mask_coexist = pos_y1 & pos_y2
Category[mask_coexist] = 1 # Yellow

# 5. プロット
fig, ax = plt.subplots(figsize=(10, 8))

# カラーマップ (同じ配色を使用)
# 1:Yellow, 2:Blue, 3:Red, 4:Gray
colors = ['white', 'gold', 'skyblue', 'salmon', '#444444']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

im = ax.imshow(Category, extent=[0, 2.0, 0, 2.0], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# 境界線 (分子=0 のライン)
# y1=0: d1 = (d2*l22*rx1)/(l21*rx2)
d1_limit = (d2 * l22 * rx1) / (l21 * rx2)
ax.axhline(y=d1_limit, color='skyblue', linestyle='--', linewidth=2, label='$y_1=0$ Boundary')

# y2=0: c1 = (c2*l12*rx1)/(l11*rx2)
c1_limit = (c2 * l12 * rx1) / (l11 * rx2)
ax.axvline(x=c1_limit, color='salmon', linestyle='--', linewidth=2, label='$y_2=0$ Boundary')

# 特異点ライン (Det_y = 0)
slope = (d2 * l11 * l22) / (c2 * l12 * l21)
x_line = np.linspace(0, 2.0, 100)
ax.plot(x_line, slope * x_line, 'w:', linewidth=2, label='Singularity')

# 凡例
legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence ($y_1, y_2 > 0$)'),
    Patch(facecolor='skyblue', edgecolor='k', label='$y_1 < 0$ (Prey 1 Extinct)'),
    Patch(facecolor='salmon', edgecolor='k', label='$y_2 < 0$ (Prey 2 Extinct)'),
    Patch(facecolor='#444444', edgecolor='k', label='Both Prey Collapse'),
]

ax.legend(handles=legend_elements, loc='upper left')
ax.set_xlabel('Conversion Efficiency $c_1$')
ax.set_ylabel('Conversion Efficiency $d_1$')
ax.set_title('Stability Regions ($c_1$ vs $d_1$)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.show()